In [1]:
import numpy as np
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, set_seed, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
if torch.cuda.is_available():
    print("CUDA is available. Using GPU.")
else:
    print("CUDA is not available. Using CPU.")

CUDA is available. Using GPU.


In [3]:
MODEL_NAME = "cointegrated/rubert-tiny2"

In [4]:
NUM_LABELS = 3  # 0=negative, 1=neutral, 2=positive

In [5]:
ds = load_dataset('./../datasets/reviews_cleaned')

In [6]:
ds

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text'],
        num_rows: 119048
    })
})

In [7]:
set(ds['train']['language'])

{'kazakh', 'other', 'russian'}

In [8]:
ds_russian = ds.filter(lambda x: x['language'] in ('russian'))

In [9]:
def is_valid(example):
    t = example["combined_text"]
    if t is None:
        return False
    if isinstance(t, float) and np.isnan(t):
        return False
    if isinstance(t, str) and t.strip() == "":
        return False
    return True

ds_russian = ds_russian.filter(is_valid)

In [10]:
ds_russian

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text'],
        num_rows: 111667
    })
})

In [11]:
def calc_sentinent(row):
    sentiment = 1
    if row['rating'] > 3.0:
        sentiment = 2
    elif row['rating'] < 3.0:
        sentiment = 0
    else:
        sentiment = 1
    return {'label': sentiment}

In [12]:
ds_russian = ds_russian.map(calc_sentinent)

In [13]:
ds_russian['train']

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text', 'label'],
    num_rows: 111667
})

In [14]:
num_negative = ds_russian['train'].filter(lambda row: row['label'] == 0).num_rows
num_neutral = ds_russian['train'].filter(lambda row: row['label'] == 1).num_rows
num_positive = ds_russian['train'].filter(lambda row: row['label'] == 2).num_rows

In [15]:
print('Number of negative reviews: {}'.format(num_negative))
print('Number of neutral reviews: {}'.format(num_neutral))
print('Number of positive reviews: {}'.format(num_positive))

Number of negative reviews: 3788
Number of neutral reviews: 3227
Number of positive reviews: 104652


In [16]:
df_russian_negative = ds_russian['train'].filter(lambda row: row['label'] == 0).shuffle(seed=42).select(range(num_negative))
df_russian_neutral = ds_russian['train'].filter(lambda row: row['label'] == 1).shuffle(seed=42).select(range(num_neutral))
df_russian_positive = ds_russian['train'].filter(lambda row: row['label'] == 2).shuffle(seed=42).select(range(num_negative)) #YES, we reduce numbers

In [17]:
df_russian_reduced = concatenate_datasets([df_russian_negative, df_russian_neutral, df_russian_positive]).shuffle(seed=42)

In [18]:
df_russian_reduced

Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'language', 'rating', 'category', 'combined_text', 'label'],
    num_rows: 10803
})

In [19]:
ds_russian_splitted = df_russian_reduced.train_test_split(test_size=0.1, seed=42)

In [20]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [21]:
def tokenize(batch):
    return tokenizer(batch['combined_text'], truncation=True, max_length=256)

In [22]:
ds_tok = ds_russian_splitted.map(tokenize, batched=True)

In [23]:
ds_tok = ds_tok.remove_columns([c for c in ds_tok["train"].column_names if c not in ("input_ids", "attention_mask", "label")])

In [24]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer,
                                        pad_to_multiple_of=8 if torch.cuda.is_available() else None
                                        )

In [25]:
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {v: k for k, v in id2label.items()}

In [26]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    # ignore_mismatched_sizes=True,  # включите, если ругается на размер классификационной головы
)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider trai

In [27]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }


In [28]:
set_seed(42)

In [29]:
args = TrainingArguments(
    output_dir="./../sentiment_bge_m3_ru_cls",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.05,
    save_steps=1000,
    eval_steps=1000,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    lr_scheduler_type="linear",  # или "cosine"
    gradient_accumulation_steps=2,  # 16*2=32 эффективный train batch
    max_grad_norm=1.0,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [30]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [31]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1 Macro
912,1.314016,0.634809,0.720629,0.705081


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.

TrainOutput(global_step=912, training_loss=1.4895015160242717, metrics={'train_runtime': 24.3456, 'train_samples_per_second': 1198.001, 'train_steps_per_second': 37.461, 'total_flos': 62294130362592.0, 'train_loss': 1.4895015160242717, 'epoch': 3.0})

In [32]:
trainer.save_model("./../sentiment_balanced_ruberta_cls/prod")
tokenizer.save_pretrained("./../sentiment_balanced_ruberta_cls/prod")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./../sentiment_balanced_ruberta_cls/prod/tokenizer_config.json',
 './../sentiment_balanced_ruberta_cls/prod/tokenizer.json')

In [34]:
pred = trainer.predict(ds_tok["test"])
val_preds = np.argmax(pred.predictions, axis=-1)

print("Metrics:", pred.metrics)
print(classification_report(ds_tok["test"]["label"], val_preds, target_names=[id2label[i] for i in range(NUM_LABELS)]))

Metrics: {'test_loss': 0.6348094344139099, 'test_accuracy': 0.7206290471785384, 'test_f1_macro': 0.7050812171980384, 'test_runtime': 0.3609, 'test_samples_per_second': 2994.974, 'test_steps_per_second': 94.199}
              precision    recall  f1-score   support

    negative       0.70      0.80      0.75       395
     neutral       0.59      0.48      0.53       319
    positive       0.83      0.84      0.84       367

    accuracy                           0.72      1081
   macro avg       0.71      0.71      0.71      1081
weighted avg       0.71      0.72      0.71      1081



In [35]:
@torch.inference_mode()
def predict_sentiment(texts, max_length=256):
    model.eval()
    device = next(model.parameters()).device
    batch = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    batch = {k: v.to(device) for k, v in batch.items()}
    out = model(**batch)
    probs = torch.softmax(out.logits, dim=-1).cpu().numpy()
    pred_ids = probs.argmax(axis=-1)
    return [
        {
            "label": model.config.id2label[int(i)],
            "scores": {model.config.id2label[j]: float(p[j]) for j in range(probs.shape[1])},
        }
        for i, p in zip(pred_ids, probs)
    ]


In [36]:
print(predict_sentiment(["Товар отличный, доставка быстрая", "Полный ужас, не советую"]))

[{'label': 'positive', 'scores': {'negative': 0.011781515553593636, 'neutral': 0.05776456743478775, 'positive': 0.9304538369178772}}, {'label': 'negative', 'scores': {'negative': 0.8113463521003723, 'neutral': 0.14619624614715576, 'positive': 0.04245733469724655}}]
